# LAM 아바타 만들기 (Colab 무료 GPU)

사진 한 장으로 LAM 3D 아바타를 만들고, 실시간 통화용 ZIP을 내보냅니다.
공식 저장소 [aigc3d/LAM](https://github.com/aigc3d/LAM)의 설치 절차를 그대로 따릅니다.

## 순서
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 셀을 위에서부터 차례로 실행 (설치는 20~40분 걸릴 수 있음)
3. 5번 셀이 출력하는 `https://….gradio.live` 링크를 열어 사진 업로드 → 드라이빙 영상 선택 → **Export ZIP** 체크 → Generate
4. 끝나면 5번 셀을 멈추고 6번 셀로 ZIP 다운로드

공개 링크는 5번 셀이 실행 중일 때만 열립니다. 다 쓰면 반드시 멈추세요.


## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 2. 설치 (Python 3.10 별도 환경)
LAM은 numpy 1.23, tensorflow 2.12 등 Python 3.10 전용 패키지를 씁니다. Colab 기본 Python과 분리합니다.

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y gcc-12 g++-12 libxi6 libxrender1 libxkbcommon0 libsm6 libgl1 > /dev/null
cd /content
pip -q install uv

# Colab's default Python is newer than LAM supports and Colab sets uv env
# vars that point at the system Python, so pin every uv call to this venv.
unset UV_SYSTEM_PYTHON UV_PYTHON UV_PROJECT_ENVIRONMENT
VENV=/content/lam310
PY=$VENV/bin/python
if [ ! -x "$PY" ] || ! "$PY" -c 'import sys; assert sys.version_info[:2] == (3, 10)'; then
  rm -rf "$VENV"
  uv venv --python 3.10 --seed "$VENV"
fi
"$PY" --version
UVPIP="uv pip install --python $PY"

# CUDA 12.1 toolkit in its own prefix (does not touch Colab's system CUDA)
if [ ! -x /content/cuda121/bin/nvcc ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
  /content/bin/micromamba create -y -q -p /content/cuda121 -c nvidia/label/cuda-12.1.1 cuda-toolkit
fi
# Build toolchain matching torch 2.3 (cu121). Colab ships a newer CUDA, and
# CUDA 12.1 rejects gcc newer than 12, so pin both.
export CUDA_HOME=/content/cuda121
export PATH=$CUDA_HOME/bin:/content/lam310/bin:$PATH
export CPATH=$CUDA_HOME/targets/x86_64-linux/include:$CUDA_HOME/include:${CPATH:-}
export LIBRARY_PATH=$CUDA_HOME/targets/x86_64-linux/lib:$CUDA_HOME/lib:${LIBRARY_PATH:-}
export LD_LIBRARY_PATH=$CUDA_HOME/targets/x86_64-linux/lib:$CUDA_HOME/lib:${LD_LIBRARY_PATH:-}
export CC=gcc-12 CXX=g++-12 CUDAHOSTCXX=g++-12
export TORCH_CUDA_ARCH_LIST=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1)
nvcc --version | tail -2
echo "TORCH_CUDA_ARCH_LIST=$TORCH_CUDA_ARCH_LIST"

[ -d LAM ] || git clone --depth 1 https://github.com/aigc3d/LAM.git
cd LAM

$UVPIP torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
$UVPIP xformers==0.0.26.post1 --index-url https://download.pytorch.org/whl/cu121
$UVPIP setuptools wheel ninja Cython numpy==1.23.0
$UVPIP --no-build-isolation -r requirements.txt

cd external/landmark_detection/FaceBoxesV2/utils/
sh make.sh
"$PY" -c 'import torch, pytorch3d, diff_gaussian_rasterization; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'
echo "INSTALL OK"

## 3. 모델 가중치와 에셋 다운로드 (Hugging Face 공식)

In [ ]:
%%bash
set -e
export PATH=/content/lam310/bin:$PATH
cd /content/LAM
huggingface-cli download 3DAIGC/LAM-assets --local-dir ./tmp
tar -xf ./tmp/LAM_assets.tar && rm ./tmp/LAM_assets.tar
tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/
huggingface-cli download 3DAIGC/LAM-20K --local-dir ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/
echo "WEIGHTS OK"

## 4. 실시간 통화용 ZIP 내보내기 도구
공식 가이드([AVATAR_EXPORT_GUIDE](https://github.com/aigc3d/LAM/blob/main/tools/AVATAR_EXPORT_GUIDE.md))의 FBX SDK 휠, Blender 4.0.2, 채팅 템플릿입니다.

In [ ]:
%%bash
set -e
unset UV_SYSTEM_PYTHON UV_PYTHON UV_PROJECT_ENVIRONMENT
PY=/content/lam310/bin/python
cd /content/LAM
wget -q -nc https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl
uv pip install --python $PY fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl pathlib patool
wget -q -nc https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
[ -d /content/blender-4.0.2-linux-x64 ] || tar -xf blender-4.0.2-linux-x64.tar.xz -C /content/
wget -q -nc https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/sample_oac.tar
tar -xf sample_oac.tar -C assets/
ls assets/sample_oac
echo "EXPORT TOOLS OK"

## 5. LAM 앱 실행
출력에 나오는 `Running on public URL: https://….gradio.live` 링크를 여세요.

- 왼쪽에 사진 업로드 (정면 사진일수록 좋음)
- 가운데 예시 영상 중 하나 선택 (얼굴 움직임을 따라 함)
- **Export ZIP file for Chatting Avatar** 체크
- Generate (약 1분)

다 끝나면 이 셀을 **멈추세요** (■ 버튼).

In [ ]:
%%bash
# Build toolchain matching torch 2.3 (cu121). Colab ships a newer CUDA, and
# CUDA 12.1 rejects gcc newer than 12, so pin both.
export CUDA_HOME=/content/cuda121
export PATH=$CUDA_HOME/bin:/content/lam310/bin:$PATH
export CPATH=$CUDA_HOME/targets/x86_64-linux/include:$CUDA_HOME/include:${CPATH:-}
export LIBRARY_PATH=$CUDA_HOME/targets/x86_64-linux/lib:$CUDA_HOME/lib:${LIBRARY_PATH:-}
export LD_LIBRARY_PATH=$CUDA_HOME/targets/x86_64-linux/lib:$CUDA_HOME/lib:${LD_LIBRARY_PATH:-}
export CC=gcc-12 CXX=g++-12 CUDAHOSTCXX=g++-12
export TORCH_CUDA_ARCH_LIST=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1)
cd /content/LAM
# Expose the app through a temporary public Gradio link (Colab has no local browser)
sed -i 's/demo.launch()/demo.launch(share=True)/' app_lam.py
/content/lam310/bin/python app_lam.py --blender_path /content/blender-4.0.2-linux-x64/blender 2>&1 | tee /content/lam_app.log

## 6. 결과 다운로드
실시간 통화용 ZIP(`output/open_avatar_chat/*.zip`)과 렌더링 영상을 내려받습니다.

In [ ]:
import glob
from google.colab import files

zips = glob.glob('/content/LAM/output/open_avatar_chat/*.zip')
videos = sorted(glob.glob('/content/LAM/**/*.mp4', recursive=True), key=lambda p: -__import__('os').path.getmtime(p))[:2]
print('ZIP:', zips)
print('VIDEO:', videos)
for path in zips + videos:
    files.download(path)